In [ ]:
# Tesla vs GameStop Stock & Revenue Analysis

In [ ]:
#Import required libraries
!pip install yfinance plotly beautifulsoup4 lxml
import pandas as pd
import requests
from bs4 import BeautifulSoup
import yfinance as yf
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
# Create ticker objects for Tesla and GameStop
tesla = yf.Ticker("TSLA")
gme = yf.Ticker("GME")

In [ ]:
# Download full historical stock price data
tesla_data = tesla.history(period="max").reset_index()
gme_data = gme.history(period="max").reset_index()

In [ ]:
# Preview stock data
print("Tesla stock data:")
print(tesla_data.head())

In [ ]:
print("\nGameStop stock data:")
print(gme_data.head())

In [ ]:
#Collect revenue data from Macrotrends webpages
# Define source URLs
tesla_url = "https://www.macrotrends.net/stocks/charts/TSLA/tesla/revenue"
gme_url = "https://www.macrotrends.net/stocks/charts/GME/gamestop/revenue"

In [ ]:
# Add headers to avoid request blocking
headers = {"User-Agent": "Mozilla/5.0"}

In [ ]:
# Send GET requests and store page HTML
tesla_html = requests.get(tesla_url, headers=headers).text
gme_html = requests.get(gme_url, headers=headers).text

In [ ]:
# create BeautifulSoup objects
tesla_soup = BeautifulSoup(tesla_html, "html.parser")
gme_soup = BeautifulSoup(gme_html, "html.parser")

In [ ]:
# Read all tables from the HTML pages
tesla_tables = pd.read_html(tesla_html)
gme_tables = pd.read_html(gme_html)

In [ ]:
tesla_revenue = tesla_tables[1]
gme_revenue = gme_tables[1]

# Rename columns for consistency
tesla_revenue.columns = ["Date", "Revenue"]
gme_revenue.columns = ["Date", "Revenue"]

In [ ]:
# Preview raw revenue data
print("\nTesla revenue data (raw):")
print(tesla_revenue.head())

print("\nGameStop revenue data (raw):")
print(gme_revenue.head())

In [ ]:
# Clean and prepare the revenue datasets

def clean_revenue_data(df):
    """
    Clean revenue dataframe by:
    - removing empty rows
    - removing dollar signs and commas
    - converting Revenue to numeric
    - converting Date to datetime
    """
    df = df.copy()

    # Remove blank and invalid revenue rows
    df = df[df["Revenue"] != ""]
    df = df[df["Revenue"] != "NaN"]
    df = df.dropna(subset=["Revenue"])

    # Remove $ and commas from Revenue column
    df["Revenue"] = df["Revenue"].astype(str)
    df["Revenue"] = df["Revenue"].str.replace(",", "", regex=False)
    df["Revenue"] = df["Revenue"].str.replace("$", "", regex=False)

    # Convert data types
    df["Revenue"] = pd.to_numeric(df["Revenue"], errors="coerce")
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

    # Drop rows with invalid values after conversion
    df = df.dropna(subset=["Date", "Revenue"])

    return df

In [ ]:
# Apply cleaning function
tesla_revenue = clean_revenue_data(tesla_revenue)
gme_revenue = clean_revenue_data(gme_revenue)

In [ ]:
# Preview cleaned revenue data
print("\nTesla revenue data (cleaned):")
print(tesla_revenue.tail())

print("\nGameStop revenue data (cleaned):")
print(gme_revenue.tail())

In [ ]:
def make_graph(stock_data, revenue_data, stock_name):
    
    # Make copies to avoid modifying original data
    stock_data = stock_data.copy()
    revenue_data = revenue_data.copy()

    # Ensure Date columns are in datetime format
    stock_data["Date"] = pd.to_datetime(stock_data["Date"], errors="coerce")
    revenue_data["Date"] = pd.to_datetime(revenue_data["Date"], errors="coerce")

In [ ]:
# Function to create stock vs revenue dashboard
def make_graph(stock_data, revenue_data, stock_name):

    # Create subplot layout with 2 rows
    fig = make_subplots(
        rows=2,
        cols=1,
        shared_xaxes=True,
        subplot_titles=(f"{stock_name} Share Price", f"{stock_name} Revenue"),
        vertical_spacing=0.12
    )

    # Add stock price line chart
    fig.add_trace(
        go.Scatter(
            x=stock_data["Date"],
            y=stock_data["Close"],
            name="Share Price"
        ),
        row=1,
        col=1
    )

    # Add revenue line chart
    fig.add_trace(
        go.Scatter(
            x=revenue_data["Date"],
            y=revenue_data["Revenue"],
            name="Revenue"
        ),
        row=2,
        col=1
    )

    # Update chart layout
    fig.update_layout(
        title=f"{stock_name} Stock Price and Revenue Dashboard",
        height=700,
        showlegend=False
    )

    # Update axis labels
    fig.update_yaxes(title_text="Share Price (USD)", row=1, col=1)
    fig.update_yaxes(title_text="Revenue (USD millions)", row=2, col=1)
    fig.update_xaxes(title_text="Date", row=2, col=1)

    # Show interactive chart
    fig.show()

In [ ]:
# Plot Tesla stock price and revenue
make_graph(tesla_data, tesla_revenue, "Tesla")

In [ ]:
#  Plot GameStop stock price and revenue
make_graph(gme_data, gme_revenue, "GameStop")